# Процесс контроля разработки

### Термины
- глобальная\локальная обработка 
- интерфейс функции
- вызов метода
- доступ к свойству
- имплементация
- архитектура
- уровни архитектуры
- OOM (out-of-memory)
- примитивы

## Описание подхода к проектированию

#### Этап 1: Проектирование архитектуры
- Составить список проблемных функций
- Проблемные функции должны включать и ML
- Определить список ML задач и основные алгоритмы
- Обсудить основные реализации глобальной обработки разных функции в разных фреймворках
- Выбрать алгоритмы для глобальной обработки
- Посмотреть на псевдокод алгоритма
- Найти проблемные места алгоритмов, переполняющие память
- Предложить архитектуру для отдельных алгоритмов
- Унифицировать архитектуру
- Дать описание работы элементов архитектуры, список функционала, список зависимостей
- Проконтролировать концепцию архитектуры с помощью чек-листа поэлементно

#### Этап 2: Тестирование архитектуры
- Для теста реализовать функцию глобальной обработки (или несколько)
- Под разработку функции создать архитектуру и проконтролировать
- Проконтролировать разработку функции с помощью чек-листа
- Тест провести на крошечном чанкованном датасете
- Установить для теста крошечный и плавающий лимит памяти, чтобы подтвердить возможность регулировки параметров обработки


*Примечание:* Можно изучить только один алгоритм глобальной обработки и больше полагаться на личные представления об архитектуре и примитивах.

## Описание подхода к контролю

- *Проектировщик* задаёт контрольные вопросы после каждого предложения *Разработчика*.
- *Разработчика* обязан отвечать на них, не отвлекаясь на код.
- Если *Разработчика* не может ответить чётко – значит, архитектура не готова.
- Только после прохождения всех контрольных вопросов *Разработчика* получает разрешение на реализацию в виде кода.

### Рекомендации *Разработчику* 
1) Формализовать этапы каждой задачи:

    - Анализ требований (что должна делать функция с точки зрения пользователя).
    - Архитектурное проектирование (на каком уровне размещается, какие примитивы использует).
    - Проектирование алгоритма (описание словами, без кода).
    - Оценка памяти (какие данные и когда хранятся, как MemoryGuard влияет на параметры).
    - Реализация (только после утверждения предыдущих этапов).

2) Ввести обязательный шаг «анти-реализация»: прежде чем писать код, нужно явно указать, какой код не будет написан (например, «Я не буду использовать список для накопления значений», «Я не буду загружать все индексы»).

3) Использовать MemoryGuard не как пассивный индикатор, а как активный регулятор:

    - Перед операцией запрашивать у MemoryGuard допустимые лимиты (размер чанка, max_keys_in_dict, buffer_sizes).
    - Если операция требует больше памяти, MemoryGuard может предложить альтернативные параметры (увеличить число партиций, уменьшить размер чанка).
    - Возможно, стоит добавить метод MemoryGuard.adjust_parameters(requested_params).

4) Покрывать каждый примитив тестом на утечку памяти с использованием memory_profiler (например, требование, что при обработке 10M строк память не превышает 2 * chunk_size * num_columns * 8 байт).

5) ⚠️ Ввести правило «ни одного цикла без ограничения по памяти»: каждый цикл, накапливающий данные, должен иметь условие сброса на диск или очистки.

6) ⚠️ *Проектировщик* не должен позволять *Разработчику*  писать код, пока не расписан детальный алгоритм на псевдокоде и не проверены его на потенциальные OOM-сценарии. Если *Разработчик* начинает писать код раньше – его нужно остановить  и требовать сначала описание.

## Управление разработкой

#### Инструкции *Разработчику:*
"Ты выполняешь роль *AI-Разработчика*. Тебе нужно дать четкое структурирование описание разрабатываемого компонента. Тебе нужно ответить на контрольные вопросы - ответ можно включить в описание или ответить отдельно. Оцени, дал ли исчерпывавшие ответы на все контрольные вопросы."

#### Структура описания:
- Цель
- Обоснование
- Алгоритм
- Наглядный пример
- Память
- Уязвимости
- Дополнительные возможности
- Регулирование потреблением памяти

#### Контрольные вопросы к *Разработчику*:
- «Что должна делать функция с точки зрения пользователя?»
- «На каком уровне размещается, какие примитивы использует?»
- «Дай описание словами, без кода.»
- «Дай простой и крошечный наглядный пример.»
- «Какой коэффициент использования памяти предполагается?»
- «Что произойдёт, если данных станет в 10 раз больше?»
- «Где в этом алгоритме данные накапливаются? Какой у них максимальный размер?»
- «Какими параметрами можно регулировать потребление памяти при работе функции?»
- «Как MemoryGuard меняет своё поведение при недостатке памяти?»
- «Что сбрасывается на диск и когда?»



⚠️ Если *Разработчик* не может ответить на любой из этих вопросов чётко – значит, **архитектура не готова**.

#### Инструкции *Разработчику:*
"Ты выполняешь роль *AI-Разработчика*.  На основании описания и ответов, которые ты дал заполни чек-лист разработки компонента. На основе заполненного чек-листа вынеси решение о возможности начала разработки компонента."

#### Чек-лист для компонента:
  1. [ ] Алгоритм описан на естественном языке, без псевдокода.
  2. [ ] Выявлены все возможные места, где память может вырасти сверх контролируемой.
  3. [ ] Предложены способы предотвращения (использование MemoryGuard для выбора параметров, сброс на диск, ограничение размера буфера).
  4. [ ] Указано, как MemoryGuard конкретно участвует – какие методы вызываются и какие параметры передаются.
  5. [ ] Написаны критерии успеха (какие тесты памяти должны пройти: например, обработка 5 млн строк с 10 колонками не превышает 200 МБ).
  6. [ ] Только после утверждения чек-листа – приступать к написанию кода.

## DONE

## Финальный список файлов проекта (после разработки примитивов)

Ниже представлены все необходимые файлы с указанием их назначения.  
Этот список соответствует утверждённой архитектуре и реализации прототипа `unique`.

```
cadw/
├── __init__.py                           # Инициализация пакета, экспорт CadwDataFrame
├── core/
│   ├── __init__.py
│   ├── memory_guard.py                   # Управление памятью (расчёт batch_size, лимитов)
│   └── spill_manager.py                  # Сброс на диск для агрегаций (уникальные, группировка)
├── io/
│   ├── __init__.py
│   ├── batch_reader.py                   # Абстрактный BatchReader + реализации (Parquet, Arrow IPC)
│   └── batch_writer.py                   # Абстрактный BatchWriter + реализации (Parquet, Arrow IPC)
├── operations/
│   ├── __init__.py
│   └── unique_collector.py               # Операция unique (с использованием SpillManager)
├── dataframe.py                          # CadwDataFrame (обёртка над источниками, метод unique)
└── tests/
    ├── __init__.py
    ├── test_memory_guard.py              # Тесты MemoryGuard
    ├── test_spill_manager.py             # Тесты SpillManager (включая сброс на диск)
    ├── test_batch_io.py                  # Тесты BatchReader/BatchWriter для Parquet и IPC
    ├── test_unique_collector.py          # Тесты UniqueCollector на малых и больших данных
    └── demo_unique.py                    # Демонстрационный скрипт (запускается отдельно)
```

### Пояснения

- **`memory_guard.py`** – класс `MemoryGuard` с методами `get_read_batch_size()`, `get_write_batch_size()`, `get_max_dict_elements()`.
- **`spill_manager.py`** – класс `SpillManager` для накопления данных (ключ → значение) и автоматического сброса на диск при превышении лимита.
- **`batch_reader.py`** – `ParquetBatchReader`, `ArrowIPCBatchReader`.
- **`batch_writer.py`** – `ParquetBatchWriter`, `ArrowIPCBatchWriter`.
- **`unique_collector.py`** – использует `BatchReader` и `SpillManager` для сбора уникальных значений.
- **`dataframe.py`** – `CadwDataFrame`, который внутри хранит список `BatchReader` (по одному на источник) и предоставляет метод `unique()`.
- **Тесты** покрывают каждый компонент изолированно.

Этот список является **окончательным** для первого этапа реализации (базовые примитивы + unique).  
Дальнейшее расширение (groupby, join) добавит новые файлы в `operations/` и, возможно, `shuffle/`.

## TO DO

################################################

In [21]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime

# --- ШАГ 1: Генерация тестового Parquet-файла с большой row group ---
print("Создаём тестовый Parquet-файл с большой row group...")

np.random.seed(42)
n_rows = 500_000  # 500 тыс. строк — достаточно для демонстрации

data = {
    'id': np.arange(n_rows),
    'timestamp': [datetime(2023, 1, 1) + pd.Timedelta(seconds=i*10) for i in range(n_rows)],
    'value': np.random.randn(n_rows),
    'category': np.random.choice(['A', 'B', 'C'], n_rows),
    'feature_1': np.random.rand(n_rows),
    'feature_2': np.random.randint(0, 100, n_rows)
}

df = pd.DataFrame(data)
table = pa.Table.from_pandas(df)

# Записываем с очень большой row group (200 000 строк) — больше, чем батч
pq.write_table(
    table,
    'test_large_rowgroup.parquet',
    row_group_size=200_000,  # row group > batch_size
    use_dictionary=True,
    compression='snappy'
)
print(f"Файл создан: test_large_rowgroup.parquet")
print(f"Размер файла: {os.path.getsize('test_large_rowgroup.parquet') / 1024 / 1024:.2f} MB")

# --- ШАГ 2: Обработка через PyArrow Dataset мини-батчами ---
print("\nНачинаем обработку через PyArrow Dataset...")

# Создаём Dataset
dataset = ds.dataset('test_large_rowgroup.parquet', format='parquet')

# Параметры
batch_size = 10_000  # Батч меньше row group (10 k < 200 k)
processed_batches = []
total_rows_processed = 0

start_time = time.time()

print(f"Итерация по батчам размером {batch_size} (row group: 200 000)...")

for batch_idx, batch in enumerate(dataset.scanner(batch_size=batch_size).to_batches()):
    print(f"Обрабатываем батч {batch_idx + 1}: {batch.num_rows} строк")
    
    # Конвертируем в Pandas для удобной обработки
    batch_df = batch.to_pandas()
    
    # ДОБАВЛЯЕМ НОВЫЕ ФИЧИ
    batch_df['new_feature_1'] = batch_df['value'] * 2 + batch_df['feature_1']
    batch_df['new_feature_2'] = np.sin(batch_df['feature_2'])
    batch_df['processed_at'] = datetime.now()
    batch_df['batch_id'] = batch_idx  # номер батча для отладки
    
    # Дополнительная обработка — например, нормализация
    batch_df['normalized_value'] = (batch_df['value'] - batch_df['value'].mean()) / batch_df['value'].std()
    
    # Обратное преобразование в PyArrow Table
    processed_batch = pa.Table.from_pandas(batch_df)
    processed_batches.append(processed_batch)
    total_rows_processed += batch.num_rows

processing_duration = time.time() - start_time
print(f"\nОбработка завершена за {processing_duration:.2f} секунд")
print(f"Всего обработано строк: {total_rows_processed}")

# --- ШАГ 3: Объединение и перезапись исходного датасета ---
print("\nОбъединяем батчи и перезаписываем исходный файл...")

# Объединяем все обработанные батчи в одну таблицу
final_table = pa.concat_tables(processed_batches)

# Перезаписываем исходный Parquet-файл (заменяем оригинал с новыми фичами)
pq.write_table(
    final_table,
    'test_large_rowgroup.parquet',  # то же имя файла — перезапись
    row_group_size=50_000,  # можно изменить размер row group
    use_dictionary=True,
    compression='snappy'
)

rewrite_time = time.time() - (start_time + processing_duration)
print(f"Перезапись завершена за {rewrite_time:.2f} секунд")

# --- ШАГ 4: Проверка результата ---
print("\nПроверяем результат...")

# Читаем обновлённый файл
updated_table = pq.read_table('test_large_rowgroup.parquet')
updated_df = updated_table.to_pandas()

print(f"Обновлённый датасет содержит {len(updated_df)} строк")
print(f"Колонки после обработки: {list(updated_df.columns)}")

# Проверяем новые фичи на примере первых 5 строк
print("\nПример новых фич (первые 5 строк):")
print(updated_df[['id', 'new_feature_1', 'new_feature_2', 'normalized_value']].head())

# --- ШАГ 5: Финальная статистика ---
print("\n" + "="*50)
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print("="*50)

original_size = os.path.getsize('test_large_rowgroup.parquet') / 1024 / 1024
print(f"Исходный размер файла: {original_size:.2f} MB")
print(f"Время обработки батчей: {processing_duration:.2f} сек")
print(f"Время перезаписи: {rewrite_time:.2f} сек")
print(f"Общее время выполнения: {time.time() - start_time:.2f} сек")
print(f"Размер батча: {batch_size}")
print(f"Размер row group: 200 000")
print(f"Количество батчей: {len(processed_batches)}")
print(f"Успешно обработаны и добавлены новые фичи в исходный датасет!")


Создаём тестовый Parquet-файл с большой row group...
Файл создан: test_large_rowgroup.parquet
Размер файла: 16.37 MB

Начинаем обработку через PyArrow Dataset...
Итерация по батчам размером 10000 (row group: 200 000)...
Обрабатываем батч 1: 10000 строк
Обрабатываем батч 2: 10000 строк
Обрабатываем батч 3: 10000 строк
Обрабатываем батч 4: 10000 строк
Обрабатываем батч 5: 10000 строк
Обрабатываем батч 6: 10000 строк
Обрабатываем батч 7: 10000 строк
Обрабатываем батч 8: 10000 строк
Обрабатываем батч 9: 10000 строк
Обрабатываем батч 10: 10000 строк
Обрабатываем батч 11: 10000 строк
Обрабатываем батч 12: 10000 строк
Обрабатываем батч 13: 10000 строк
Обрабатываем батч 14: 10000 строк
Обрабатываем батч 15: 10000 строк
Обрабатываем батч 16: 10000 строк
Обрабатываем батч 17: 10000 строк
Обрабатываем батч 18: 10000 строк
Обрабатываем батч 19: 10000 строк
Обрабатываем батч 20: 10000 строк
Обрабатываем батч 21: 10000 строк
Обрабатываем батч 22: 10000 строк
Обрабатываем батч 23: 10000 строк
Обрабат

In [22]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pandas as pd
import numpy as np
import time

# Параметры
input_filename = '.OTHER/files/test_large_rowgroup.parquet'
output_filename = '.OTHER/files/processed_streamed.parquet'
batch_size = 10_000  # Батч меньше row group (500 000)


# Создаём Dataset для чтения
dataset = ds.dataset(input_filename, format='parquet')


# Итератор по батчам
batches_iterator = dataset.scanner(batch_size=batch_size).to_batches()

# Читаем первый батч для определения исходной схемы
first_batch = next(batches_iterator)
first_batch_df = first_batch.to_pandas()


# Добавляем новые фичи к первому батчу
first_batch_df['new_feature_1'] = first_batch_df['value'] * 2 + first_batch_df['feature_1']
first_batch_df['new_feature_2'] = np.sin(first_batch_df['feature_2'])
first_batch_df['processed_at'] = pd.Timestamp.now()
first_batch_df['batch_id'] = 0
first_batch_df['normalized_value'] = (first_batch_df['value'] - first_batch_df['value'].mean()) / first_batch_df['value'].std()


# Создаём итоговую схему с новыми колонками
processed_first_batch = pa.Table.from_pandas(first_batch_df)
output_schema = processed_first_batch.schema

# Создаём ParquetWriter с явной схемой
with pq.ParquetWriter(
    output_filename,
    schema=output_schema,
    compression='snappy',
    use_dictionary=True
) as writer:
    # Записываем первый обработанный батч
    writer.write_table(processed_first_batch)
    total_rows_processed = first_batch.num_rows
    print(f"Обработан и записан первый батч: {first_batch.num_rows} строк")

    # Обрабатываем остальные батчи
    batch_idx = 1
    for batch in batches_iterator:
        print(f"Обрабатываем батч {batch_idx + 1}: {batch.num_rows} строк")
        # Конвертируем в Pandas для удобной обработки
        batch_df = batch.to_pandas()
        # Добавляем те же новые фичи
        batch_df['new_feature_1'] = batch_df['value'] * 2 + batch_df['feature_1']
        batch_df['new_feature_2'] = np.sin(batch_df['feature_2'])
        batch_df['processed_at'] = pd.Timestamp.now()
        batch_df['batch_id'] = batch_idx
        # Нормализация (по статистике текущего батча)
        batch_df['normalized_value'] = (batch_df['value'] - batch_df['value'].mean()) / batch_df['value'].std()
        # Обратное преобразование в PyArrow Table
        processed_batch = pa.Table.from_pandas(batch_df)
        # Записываем обработанный батч в выходной файл
        writer.write_table(processed_batch)
        total_rows_processed += batch.num_rows
        batch_idx += 1

print(f"\nВсего обработано строк: {total_rows_processed}")
full_dataset = ds.dataset('.OTHER/files/', format='parquet')


Обработан и записан первый батч: 10000 строк
Обрабатываем батч 2: 10000 строк
Обрабатываем батч 3: 10000 строк
Обрабатываем батч 4: 10000 строк
Обрабатываем батч 5: 10000 строк
Обрабатываем батч 6: 10000 строк
Обрабатываем батч 7: 10000 строк
Обрабатываем батч 8: 10000 строк
Обрабатываем батч 9: 10000 строк
Обрабатываем батч 10: 10000 строк
Обрабатываем батч 11: 10000 строк
Обрабатываем батч 12: 10000 строк
Обрабатываем батч 13: 10000 строк
Обрабатываем батч 14: 10000 строк
Обрабатываем батч 15: 10000 строк
Обрабатываем батч 16: 10000 строк
Обрабатываем батч 17: 10000 строк
Обрабатываем батч 18: 10000 строк
Обрабатываем батч 19: 10000 строк
Обрабатываем батч 20: 10000 строк
Обрабатываем батч 21: 10000 строк
Обрабатываем батч 22: 10000 строк
Обрабатываем батч 23: 10000 строк
Обрабатываем батч 24: 10000 строк
Обрабатываем батч 25: 10000 строк
Обрабатываем батч 26: 10000 строк
Обрабатываем батч 27: 10000 строк
Обрабатываем батч 28: 10000 строк
Обрабатываем батч 29: 10000 строк
Обрабатывае

ArrowInvalid: Error creating dataset. Could not read schema from '.OTHER/files/processed.lance/data/001001101011011111011001fc634444c4a84a70274ff614ad.lance'. Is this a 'parquet' file?: Could not open Parquet input source '.OTHER/files/processed.lance/data/001001101011011111011001fc634444c4a84a70274ff614ad.lance': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
import lance
import pyarrow as pa

# 1. Создаём тестовый датасет
data = pa.table({
    "id": range(100_000),
    "vector": [[i, i+1] for i in range(100_000)],
    "category": ["A" if i % 2 == 0 else "B" for i in range(100_000)]
})
lance.write_dataset(data, "test.lance")

# 2. Загружаем как Lance Dataset
dataset = lance.dataset("test.lance")

# 3. Доступ по индексу (строка № 50 000)
single_row = dataset.take([50_000])
print(single_row)

# 4. Фильтрация с индексом (быстро, т. к. есть индекс по 'id')
filtered = dataset.scanner(filter="id == 99_999").to_table()
print(filtered)

# 5. Векторный поиск (10 ближайших к [50_000, 50_001])
nearest = dataset.scanner(
    nearest="vector",
    query=[50_000.0, 50_001.0],
    k=10
).to_table()
print(nearest)


pyarrow.Table
id: int64
vector: list<item: int64>
  child 0, item: int64
category: string
----
id: [[50000]]
vector: [[[50000,50001]]]
category: [["A"]]
pyarrow.Table
id: int64
vector: list<item: int64>
  child 0, item: int64
category: string
----
id: [[99]]
vector: [[[99,100]]]
category: [["B"]]


TypeError: LanceDataset.scanner() got an unexpected keyword argument 'query'

In [ ]:
import lance
import pandas as pd
import pyarrow as pa
import pyarrow.dataset

df = pd.DataFrame({"a": [5], "b": [10]})
uri_parquet = "/tmp/test.parquet"
tbl = pa.Table.from_pandas(df)
pa.dataset.write_dataset(tbl, uri_parquet, format='parquet')

parquet_ds = pa.dataset.dataset(uri_parquet, format='parquet')
lance.write_dataset(parquet_ds, "/tmp/test.lance")


In [ ]:
dataset = lance.dataset("/tmp/test.lance")
# Теперь `dataset` — это объект PyArrow Dataset, с которым можно работать
# Например, через Pandas:
df = dataset.to_table().to_pandas()
print(df)

# # Или через DuckDB:
# import duckdb
# duckdb.query("SELECT * FROM dataset LIMIT 10").to_df()


   a   b
0  5  10


In [ ]:
import lance
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import numpy as np
import os

# Параметры
input_parquet = '.OTHER/files/test_large_rowgroup.parquet'
output_lance = '.OTHER/files/processed.lance'
batch_size = 10_000  # Батч меньше row group

def add_features(df):
    """Функция добавления новых фич к DataFrame"""
    df['new_feature_1'] = df['value'] * 2 + df['feature_1']
    df['new_feature_2'] = np.sin(df['feature_2'])
    df['processed_at'] = pd.Timestamp.now()
    df['batch_id'] = batch_idx
    # Нормализация по статистике батча
    df['normalized_value'] = (df['value'] - df['value'].mean()) / df['value'].std()
    return df

# Создаём PyArrow Dataset из Parquet
parquet_dataset = ds.dataset(input_parquet, format='parquet')

# Итератор по батчам
scanner = parquet_dataset.scanner(batch_size=batch_size)
batches = scanner.to_batches()

# Обрабатываем первый батч для определения схемы
first_batch = next(batches)
first_df = first_batch.to_pandas()
first_processed = add_features(first_df)

# Записываем первый батч в Lance (создаёт файл)
lance.write_dataset(
    pa.Table.from_pandas(first_processed),
    output_lance
)
total_rows = len(first_processed)
print(f"Создан Lance датасет, добавлен первый батч: {total_rows} строк")

# Открываем существующий Lance датасет для добавления данных
existing_dataset = lance.dataset(output_lance)

# Обрабатываем остальные батчи и добавляем в Lance датасет
batch_idx = 1
for batch in batches:
    batch_df = batch.to_pandas()
    processed_batch = add_features(batch_df)
    # Добавляем обработанный батч в существующий Lance датасет
    existing_dataset.add(pa.Table.from_pandas(processed_batch))
    total_rows += len(processed_batch)
    print(f"Добавлен батч {batch_idx + 1}: {len(processed_batch)} строк, всего: {total_rows}")
    batch_idx += 1

print(f"\nВсего обработано и записано строк: {total_rows}")


Создан Lance датасет, добавлен первый батч: 10000 строк


AttributeError: 'LanceDataset' object has no attribute 'add'

In [8]:
import tempfile
import time

import tempfile

temp_dir = tempfile.mkdtemp(prefix='safe_')
print(f'Безопасная директория: {temp_dir}')

# # Установить кастомную директорию
# tempfile.tempdir = './temp/folder'

# # Получить текущую временную директорию
# print(tempfile.gettempdir())


# Создаём временный файл
temp = tempfile.TemporaryFile()
time.sleep(30)

print(temp)  # <_io.BufferedRandom name=7>
print(temp.name)  # 7 (не реальное имя)

# Запись данных (в бинарном режиме)
temp.write(b'Hello, World!')
temp.seek(0)  # Перемещаем указатель в начало

# Чтение данных
print(temp.read())  # b'Hello, World!'

temp.close()  # Файл удаляется автоматически


Безопасная директория: /tmp/safe_zqc0_7d3
<_io.BufferedRandom name=68>
68
b'Hello, World!'


In [ ]:
# tests/demo_unique.py
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import tempfile
import time
import os
from cadw.core.memory_guard import MemoryGuard
from cadw.io.batch_reader import ParquetBatchReader
from cadw.operations.unique_collector import UniqueCollector

def generate_test_data(parquet_path, num_rows=200_000, num_unique=50_000):
    np.random.seed(42)
    unique_vals = list(range(num_unique))
    data = np.array(unique_vals * (num_rows // num_unique))[:num_rows]
    table = pa.table({'col': data})
    pq.write_table(table, parquet_path)
    print(f"Сгенерирован {parquet_path} с {num_rows} строк, {num_unique} уникальных.")

def main():
    # Создаём тестовый файл
    with tempfile.TemporaryDirectory() as tmpdir:
        input_path = os.path.join(tmpdir, 'test.parquet')
        generate_test_data(input_path, num_rows=200_000, num_unique=50_000)
        time.sleep(10)

        # Ограничение памяти 10 MB – вынуждаем спилл
        guard = MemoryGuard(10)
        reader = ParquetBatchReader(input_path, ['col'], guard.get_read_batch_size())
        print('batch_size:', guard.get_read_batch_size())
        collector = UniqueCollector(reader, 'col', guard)
        uniq = collector.collect()

        print(f"Найдено уникальных значений: {len(uniq)}")
        assert len(uniq) == 50_000
        print("Тест пройден!")

if __name__ == '__main__':
    main()

Сгенерирован /tmp/tmprknffuax/test.parquet с 200000 строк, 50000 уникальных.
batch_size: 13107
Найдено уникальных значений: 50000
Тест пройден!
